# DUCKDB

Since data from EuroCropsV2 is in duckdb format, i will try to load everything in a dataframe, analyze a portion of all the data that there is and then try to figure out how to concatenate with the satellites photos during a given timeframe eg. a year or so


In [2]:
import duckdb
import datetime as dt

from EuroCropsV2.code.import_db.a00_main import *
from EuroCropsV2.code.utils.tools import global_config

start_time = '01-01-2021'
end_time = '31-12-2022'

main(global_config)


ModuleNotFoundError: No module named 'code.utils'; 'code' is not a package

In [1]:
# import tempfile
# tempfile.tempdir = '/scratch/clamart/'

# import json
import duckdb
import datetime as dt
import ipywidgets

RunDownload  = True
RunImportDuckDB = True
RunImportPGDB = False



# 

import os
from EuroCropsV2.code.utils.tools import global_config

# import pdb;set_trace()

os.makedirs(os.path.dirname(os.path.abspath(global_config.paths['duckdbpath'])), exist_ok=True)

# dd
def main(global_config):
    if RunDownload:
        print("Downloading files from FTP")
        from EuroCropsV2.code.import_db.a01_download_from_ftp import download as download_from_ftp
        download_from_ftp(global_config)
    if RunImportDuckDB:
        print("Importing files to DuckDB")
        from EuroCropsV2.code.import_db.a02_import_to_duckdb import upload as import_to_duckdb
        import_to_duckdb(global_config)
    if RunImportPGDB:
        print("Importing files to PostgreSQL/PostGIS")
        from EuroCropsV2.code.import_db.a03_import_to_pgdb import upload as import_to_pgdb
        import_to_pgdb(global_config)
        # run 04_update_mapping_tables.py
        # run 05_Create_GSA_all_view_layers.py
        # run 06_create_gsa_crop_grid_2.py
    


main(global_config)


['at_2020.parquet', 'at_stack.parquet', 'be2_2020.parquet', 'be2_stack.parquet', 'be3_2020.parquet', 'be3_stack.parquet', 'bg_2020.parquet', 'bg_stack.parquet', 'cz_stack.parquet', 'de4_2020.parquet', 'de4_stack.parquet', 'de4_stack_bu.parquet', 'dea_2020.parquet', 'dea_stack.parquet', 'dk_2020.parquet', 'dk_stack.parquet', 'ee_2020.parquet', 'ee_stack.parquet', 'es_stack.parquet', 'fi_2020.parquet', 'fi_stack.parquet', 'fr_2020.parquet', 'fr_stack.parquet', 'ie_2020.parquet', 'ie_stack.parquet', 'iti1_2020.parquet', 'iti1_stack.parquet', 'nl_2020.parquet', 'nl_stack.parquet', 'pt_2020.parquet', 'pt_stack.parquet', 'si_2020.parquet', 'si_stack.parquet', 'sk_2020.parquet', 'sk_stack.parquet']
Found 35 .parquet files. Starting download...
Skipping (exists): at_2020.parquet
Skipping (exists): at_stack.parquet
Skipping (exists): be2_2020.parquet
Skipping (exists): be2_stack.parquet
Skipping (exists): be3_2020.parquet
Skipping (exists): be3_stack.parquet
Skipping (exists): bg_2020.parquet
S

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DuckDB table 'eurocrops' written using native API.
DuckDB table 'hcat4_agriprod_mapping' written using native API.
DuckDB table 'hcat4_eagle_mapping' written using native API.
DuckDB table 'hcat4_hrl_mapping' written using native API.
DuckDB table 'agriprod_fadn_mapping' written using native API.
DuckDB table 'hcat4' written using native API.
-----> exec time = 0.00mn


In [2]:
conn = duckdb.connect(global_config.paths['duckdbpath'])

tables_df = conn.execute("SHOW TABLES").fetchdf()
print(tables_df)

                     name
0   agriprod_fadn_mapping
1               eurocrops
2                   hcat4
3  hcat4_agriprod_mapping
4     hcat4_eagle_mapping
5       hcat4_hrl_mapping


In [3]:
conn.execute("SELECT * FROM agriprod_fadn_mapping").fetch_df()

,agriprod_code,agriprod_name,fadn_code,code_name
0,ARA,Arable land,SE026,Arable land (ha)
1,C0000,Cereals for the production of grain (including...,SE035,Cereals (ha)
2,C1000,Cereals (excluding rice) for the production of...,SE035,Cereals (ha)
3,C1110,Common wheat and spelt,SE035,Cereals (ha)
4,C1111,Common winter wheat and spelt,SE035,Cereals (ha)
...,...,...,...,...
183,W1000,Grapes,SE050,Vineyards (ha)
184,W1100,Grapes for wines,SE050,Vineyards (ha)
185,W1200,Grapes for table use,SE050,Vineyards (ha)
186,W1300,Grapes for raisins,SE050,Vineyards (ha)


In [5]:



test = conn.execute("SET enable_geoparquet_conversion = false; SELECT * FROM eurocrops " \
"INNER JOIN hcat4 on eurocrops.hcat4_code = hcat4.hcat4_code " \
"INNER JOIN hcat4_agriprod_mapping as hcat4_am on eurocrops.hcat4_code = hcat4_am.hcat4_code " \
"INNER JOIN hcat4_eagle_mapping as hcat4_em on eurocrops.hcat4_code = hcat4_em.hcat4_code " \
"INNER JOIN hcat4_hrl_mapping as hcat4_hrl on eurocrops.hcat4_code = hcat4_hrl.hcat4_code " \
"INNER JOIN agriprod_fadn_mapping as agfm on hcat4_am.agriprod_code = agfm.agriprod_code").fetch_df()

InvalidIndexError: (slice(None, 5, None), slice(None, 5, None))

In [8]:
test.columns.values

array(['nuts', 'original_code', 'original_name', 'translated_name',
       'hcat4_code', 'hcat4_name', 'usage_code', 'usage_name',
       'hcat4_code_1', 'hcat4_name_1', 'seasonality_code',
       'seasonality_name', 'hcat4_code_2', 'hcat4_name_2', 'usage_code_1',
       'usage_name_1', 'link', 'agriprod_code', 'agriprod_name',
       'hcat4_code_3', 'hcat4_name_3', 'link_1', 'eagle_code',
       'eagle_name', 'eagle_seasonality_code', 'eagle_seasonality_name',
       'hcat4_code_4', 'hcat4_name_4', 'hrl_code', 'hrl_name',
       'agriprod_code_1', 'agriprod_name_1', 'fadn_code', 'code_name'],
      dtype=object)

In [42]:
conn = duckdb.connect(global_config.paths["duckdbpath"])

tables = conn.execute("SHOW TABLES").fetchdf()
print(tables)

                     name
0   agriprod_fadn_mapping
1               eurocrops
2                   hcat4
3  hcat4_agriprod_mapping
4     hcat4_eagle_mapping
5       hcat4_hrl_mapping


In [45]:
conn_shady = duckdb.connect('/home/wr3nch/Documents/Projects/RSSIA/dataset/exported/eurocropsv2.duckdb')

conn_shady.execute("SHOW TABLES").fetch_df()

,name
0,agriprod_fadn_mapping
1,eurocrops
2,hcat4
3,hcat4_agriprod_mapping
4,hcat4_eagle_mapping
5,hcat4_hrl_mapping


In [49]:
import duckdb

file_path = (
    "/home/wr3nch/Documents/Projects/RSSIA/"
    "dataset/at_2020.parquet"
)

conn = duckdb.connect()

conn.execute("SET enable_geoparquet_conversion = false;")

df = conn.execute("""
    SELECT *
    FROM read_parquet(?)
    LIMIT 10
""", [file_path]).fetchdf()

df

,cropfield,original_code,off_id,off_area,area_ha,geometry
0,1056692,635,84429120.0,0.109817,0.109879,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
1,18906,708,83405488.0,1.807834,1.808831,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
2,18907,707,83405461.0,0.020992,0.021004,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
3,18908,716,83405851.0,0.069894,0.069935,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
4,18909,990,83393545.0,1.535157,1.536022,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
5,18910,716,83393514.0,1.037554,1.038136,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
6,18912,307,83395160.0,3.731566,3.734082,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
7,18913,110,83395163.0,0.801534,0.802073,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
8,18939,715,83405649.0,0.995825,0.996402,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
9,18940,717,83405659.0,0.360944,0.361152,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."


# AI Generated code
This section contains ai generated code because i don't get how the geometry (lat and long) is not loaded by their scripts but it is actually present in the parquet files.  

In [13]:
from pathlib import Path
import duckdb
import pandas as pd


# ------------------------------------------------------------------
# 1. Paths and table name
# ------------------------------------------------------------------

parquet_path = Path(
    "/home/wr3nch/Documents/Projects/RSSIA/dataset/at_2020.parquet"
)

duckdb_path = Path(
    "/home/wr3nch/Documents/Projects/RSSIA/dataset/eurocrops.duckdb"
)

table_name = "at_2020"


# ------------------------------------------------------------------
# 2. Validate paths
# ------------------------------------------------------------------

if not parquet_path.exists():
    raise FileNotFoundError(
        f"Parquet file does not exist:\n{parquet_path}"
    )

duckdb_path.parent.mkdir(parents=True, exist_ok=True)

print("Parquet file:", parquet_path)
print("DuckDB file:", duckdb_path)
print("DuckDB table:", table_name)


# ------------------------------------------------------------------
# 3. Open DuckDB
# ------------------------------------------------------------------

conn = duckdb.connect(str(duckdb_path))


# ------------------------------------------------------------------
# 4. Load the spatial extension
# ------------------------------------------------------------------

try:
    conn.execute("LOAD spatial")
except duckdb.Error:
    # INSTALL requires internet only if the extension is not already installed.
    conn.execute("INSTALL spatial")
    conn.execute("LOAD spatial")


# ------------------------------------------------------------------
# 5. Disable automatic GeoParquet conversion
#
# The EuroCropsV2 files can contain incomplete GeoParquet metadata.
# With conversion disabled, geometry is read as raw WKB/BLOB.
# ------------------------------------------------------------------

conn.execute("""
    SET enable_geoparquet_conversion = false
""")


# ------------------------------------------------------------------
# 6. Inspect the raw Parquet schema
# ------------------------------------------------------------------

raw_schema = conn.execute(
    """
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
    """,
    [str(parquet_path)],
).fetchdf()

print("\nRaw Parquet schema:")
display(raw_schema)


# ------------------------------------------------------------------
# 7. Verify that the geometry column exists
# ------------------------------------------------------------------

raw_columns = set(raw_schema["column_name"].str.lower())

if "geometry" not in raw_columns:
    raise RuntimeError(
        "The Parquet file does not contain a column named 'geometry'. "
        f"Available columns: {sorted(raw_columns)}"
    )


# ------------------------------------------------------------------
# 8. Import the Parquet file into DuckDB
# ------------------------------------------------------------------

conn.execute(
    f"""
    CREATE OR REPLACE TABLE "{table_name}" AS
    SELECT *
    FROM read_parquet(?)
    """,
    [str(parquet_path)],
)

print(f"\nImported {parquet_path.name} into table {table_name!r}")


# ------------------------------------------------------------------
# 9. Convert raw WKB geometry to DuckDB GEOMETRY
# ------------------------------------------------------------------

conn.execute(
    f"""
    ALTER TABLE "{table_name}"
    ADD COLUMN geom GEOMETRY
    """
)

conn.execute(
    f"""
    UPDATE "{table_name}"
    SET geom = ST_GeomFromWKB(geometry)
    WHERE geometry IS NOT NULL
    """
)


# ------------------------------------------------------------------
# 10. Remove the original raw WKB column
#
# Comment this section out if you want to preserve both geometry and geom.
# ------------------------------------------------------------------

conn.execute(
    f"""
    ALTER TABLE "{table_name}"
    DROP COLUMN geometry
    """
)


# ------------------------------------------------------------------
# 11. Create a spatial index
# ------------------------------------------------------------------

conn.execute(
    f"""
    CREATE INDEX IF NOT EXISTS "{table_name}_geom_rtree"
    ON "{table_name}"
    USING RTREE (geom)
    """
)


# ------------------------------------------------------------------
# 12. Verify the imported table
# ------------------------------------------------------------------

imported_schema = conn.execute(
    f'DESCRIBE "{table_name}"'
).fetchdf()

print("\nImported DuckDB schema:")
display(imported_schema)

row_count = conn.execute(
    f'SELECT COUNT(*) FROM "{table_name}"'
).fetchone()[0]

geometry_count = conn.execute(
    f"""
    SELECT COUNT(*)
    FROM "{table_name}"
    WHERE geom IS NOT NULL
    """
).fetchone()[0]

print("\nTotal rows:", row_count)
print("Rows with geometry:", geometry_count)


# ------------------------------------------------------------------
# 13. Check geometry types
# ------------------------------------------------------------------

geometry_types = conn.execute(
    f"""
    SELECT
        ST_GeometryType(geom) AS geometry_type,
        COUNT(*) AS row_count
    FROM "{table_name}"
    WHERE geom IS NOT NULL
    GROUP BY ST_GeometryType(geom)
    ORDER BY row_count DESC
    """
).fetchdf()

print("\nGeometry types:")
display(geometry_types)


# ------------------------------------------------------------------
# 14. Derive longitude and latitude
#
# EuroCropsV2 geometry is expected to use EPSG:3035.
#
# ST_PointOnSurface returns a representative point guaranteed to lie
# inside the parcel. The point is then transformed to EPSG:4326.
#
# X = longitude
# Y = latitude
# ------------------------------------------------------------------

coordinate_query = f"""
WITH representative_points AS (
    SELECT
        *,
        ST_PointOnSurface(geom) AS point_3035
    FROM "{table_name}"
    WHERE geom IS NOT NULL
),

wgs84_points AS (
    SELECT
        *,
        ST_Transform(
            point_3035,
            'EPSG:3035',
            'EPSG:4326',
            always_xy := true
        ) AS point_wgs84
    FROM representative_points
)

SELECT
    * EXCLUDE (point_3035, point_wgs84),

    ST_X(point_wgs84) AS longitude,
    ST_Y(point_wgs84) AS latitude

FROM wgs84_points
"""

df = conn.execute(coordinate_query).fetchdf()


# ------------------------------------------------------------------
# 15. Inspect the resulting Pandas DataFrame
# ------------------------------------------------------------------

print("\nDataFrame shape:", df.shape)
display(df.head())

print("\nCoordinate sample:")
display(
    df[["longitude", "latitude"]]
    .dropna()
    .head(10)
)


# ------------------------------------------------------------------
# 16. Basic coordinate sanity check
# ------------------------------------------------------------------

invalid_coordinates = df[
    ~df["longitude"].between(-180, 180)
    | ~df["latitude"].between(-90, 90)
]

print("\nRows with invalid longitude/latitude:", len(invalid_coordinates))


# ------------------------------------------------------------------
# 17. Optional: export the result to Parquet
# ------------------------------------------------------------------

output_path = parquet_path.with_name("at_2020_with_coordinates.parquet")

conn.execute(
    f"""
    COPY (
        {coordinate_query}
    )
    TO ?
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """,
    [str(output_path)],
)

print("\nOutput written to:", output_path)


# ------------------------------------------------------------------
# 18. Show all tables currently in the DuckDB database
# ------------------------------------------------------------------

print("\nDuckDB tables:")
display(conn.execute("SHOW TABLES").fetchdf())


# Close only when you have finished querying the database.
# conn.close()

Parquet file: /home/wr3nch/Documents/Projects/RSSIA/dataset/at_2020.parquet
DuckDB file: /home/wr3nch/Documents/Projects/RSSIA/dataset/eurocrops.duckdb
DuckDB table: at_2020

Raw Parquet schema:


,column_name,column_type,null,key,default,extra
0,cropfield,BIGINT,YES,None,None,None
1,original_code,VARCHAR,YES,None,None,None
2,off_id,DOUBLE,YES,None,None,None
3,off_area,DOUBLE,YES,None,None,None
4,area_ha,DOUBLE,YES,None,None,None
5,geometry,BLOB,YES,None,None,None


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Imported at_2020.parquet into table 'at_2020'


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

,longitude,latitude
0,14.951756,48.585509
1,14.952288,48.572463
2,14.953267,48.579837
3,15.832805,48.401275
4,10.741907,46.989084
5,10.750154,47.046045
6,14.487506,47.090390
7,12.941125,46.645154
8,12.890411,46.687188
9,12.883620,46.687084



Rows with invalid longitude/latitude: 13


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Output written to: /home/wr3nch/Documents/Projects/RSSIA/dataset/at_2020_with_coordinates.parquet

DuckDB tables:


,name
0,at_2020
